In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [21]:
vector_add = r"""

#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h>

__global__ void vec_add(const float* a, const float* b, float* c, int n){
    int i = blockIdx.x*blockDim.x+threadIdx.x;
    if (i<n){
        c[i]=a[i]+b[i];
    }
}

int main(){
const int n=1<<20;
size_t bytes = n*sizeof(float);
float *a, *b, *c;

cudaMallocManaged(&a, bytes);
cudaMallocManaged(&b, bytes);
cudaMallocManaged(&c, bytes);

for(int i=0; i<n; i++) {a[i]=i; b[i]=2.0f*i;}

int threads=256;
int blocks=(n+threads-1)/threads;
vec_add<<<blocks, threads>>>(a, b, c, n);
cudaDeviceSynchronize();

for(int i=0; i<10; i++){
printf("c%d = %f\n",i, c[i]);
}

cudaFree(a); cudaFree(b); cudaFree(c);
}

"""

with open('vector_add.cu', 'w') as f:
    f.write(vector_add)

In [22]:
!ls

vec_add  vector_add.cu


In [23]:
!nvcc -arch=sm_75 vector_add.cu -o vec_add

In [24]:
!ls

vec_add  vector_add.cu


In [25]:
!./vec_add

c0 = 0.000000
c1 = 3.000000
c2 = 6.000000
c3 = 9.000000
c4 = 12.000000
c5 = 15.000000
c6 = 18.000000
c7 = 21.000000
c8 = 24.000000
c9 = 27.000000
